# 01 — Fashion-MNIST: tải và kiểm tra dataset

Notebook Julia độc lập để tải dataset chính thức, kiểm tra checksum, đọc định dạng IDX và xác nhận train/test split. Chạy cell theo thứ tự bằng Julia ≥ 1.9.

Nguồn: [Fashion-MNIST của Zalando](https://github.com/zalandoresearch/fashion-mnist), cũng được ghi trong `docs/plan.md`.

## 1. Kích hoạt môi trường của repository

Notebook tìm thư mục gốc từ thư mục đang mở, kích hoạt `Project.toml` tại đó và cài các dependencies đã khóa trong `Manifest.toml` nếu thiếu.

In [1]:
using Pkg

function find_repo_root(start=pwd())
    directory = abspath(start)
    while true
        if isfile(joinpath(directory, "Project.toml")) &&
           isfile(joinpath(directory, "docs", "plan.md")) &&
           isdir(joinpath(directory, "src"))
            return directory
        end
        parent = dirname(directory)
        parent == directory && error("Không tìm thấy thư mục gốc repository.")
        directory = parent
    end
end

repo_root = find_repo_root()
Pkg.activate(repo_root)
Pkg.instantiate()

using EclatFIM
using SHA: sha256
using TOML

data_dir = joinpath(repo_root, "data", "raw", "fashion-mnist")
results_dir = joinpath(repo_root, "results", "fashion")
@assert VERSION >= v"1.9"
println("Julia ", VERSION)
println("Project: ", Base.active_project())
println("Raw data: ", data_dir)


  Activating 

project at `/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining`


Julia 1

.13.3
Project: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/Project.toml
Raw data: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/data/raw/fashion-mnist


## 2. Tải bốn archive và kiểm tra SHA-256

Mỗi file được đối chiếu với checksum cố định trong source. File tải lỗi hoặc sai nội dung sẽ không được giữ làm bản dataset hợp lệ. File đã có được xác minh lại trước khi sử dụng.

In [2]:
archive_paths = download_fashion_mnist(data_dir)
expected_files = EclatFIM.FASHION_FILES

for (archive, expected) in zip(archive_paths, expected_files)
    actual = bytes2hex(sha256(read(archive)))
    @assert actual == expected.sha256
    println("SHA-256 OK  ", expected.name, "  (", filesize(archive), " bytes)")
end
println("Đã xác minh đủ 4 archive từ ", EclatFIM.FASHION_SOURCE)


SHA-256 OK  train-images-idx3-ubyte.gz

  (26421880 bytes)
SHA-256 OK  train-labels-idx1-ubyte.gz  (29515 bytes)
SHA-256 OK  t10k-images-idx3-ubyte.gz  (4422102 bytes)
SHA-256 OK  t10k-labels-idx1-ubyte.gz  (5148 bytes)
Đã xác minh đủ 4 archive từ https://github.com/zalandoresearch/fashion-mnist


## 3. Đọc ảnh và nhãn IDX

IDX lưu ảnh theo thứ tự dòng. Hàm đọc chuyển đúng về mảng Julia `images[row, column, image]`, không xoay ảnh; cường độ pixel vẫn là `UInt8` trong khoảng 0–255. Nhãn lưu tách khỏi ảnh.

In [3]:
train = load_fashion_mnist(data_dir; split=:train)
test = load_fashion_mnist(data_dir; split=:test)

println("Train: ", size(train.images), " ảnh; ", length(train.labels), " nhãn")
println("Test:  ", size(test.images), " ảnh; ", length(test.labels), " nhãn")
println("Kiểu pixel: ", eltype(train.images), "; giá trị train: ", extrema(train.images))


Train: (

28, 28, 60000) ảnh; 60000 nhãn
Test:  (28, 28, 10000) ảnh; 10000 nhãn
Kiểu pixel: UInt8

; giá trị train: (0x00, 0xff)


## 4. Xác nhận shape, nhãn và train/test split

Các assertion xác nhận đủ 60.000 ảnh train và 10.000 ảnh test; cả hai split có ảnh 28×28, nhãn 0–9 và 6.000/1.000 ảnh mỗi lớp.

In [4]:
@assert size(train.images) == (28, 28, 60_000)
@assert size(test.images) == (28, 28, 10_000)
@assert eltype(train.images) == UInt8 && eltype(test.images) == UInt8
@assert length(train.labels) == size(train.images, 3)
@assert length(test.labels) == size(test.images, 3)
@assert all(label -> 0 <= label <= 9, train.labels)
@assert all(label -> 0 <= label <= 9, test.labels)
@assert sort(unique(train.labels)) == collect(0:9)
@assert sort(unique(test.labels)) == collect(0:9)

train_counts = [count(==(label), train.labels) for label in 0:9]
test_counts = [count(==(label), test.labels) for label in 0:9]
@assert all(==(6_000), train_counts)
@assert all(==(1_000), test_counts)

println("PASS: đúng shape, grayscale UInt8, nhãn 0–9 và split train/test.")
for label in 0:9
    println(rpad("$(label) $(FASHION_CLASSES[label + 1])", 22),
        "train=", rpad(string(train_counts[label + 1]), 7),
        "test=", test_counts[label + 1])
end


PASS: đúng shape, grayscale UInt8, nhãn 0–9 và split train/test.


0 T-shirt/top         train=6000   test=1000
1 Trouser             train=6000   test=1000
2 Pullover            train=6000   test=1000
3 Dress               train=6000   test=1000
4 Coat                train=6000   test=1000
5 Sandal              train=6000   test=1000
6 Shirt               train=6000   test=1000
7 Sneaker             train=6000   test=1000
8 Bag                 train=6000   test=1000
9 Ankle boot          train=6000   test=1000


## 5. Lưu báo cáo xác minh

Báo cáo ghi nguồn, phiên bản Julia, số ảnh theo split và checksum. Các mảng ảnh không được ghi vào notebook nên file vẫn gọn.

In [5]:
mkpath(results_dir)
report = Dict(
    "dataset" => "Fashion-MNIST",
    "source" => EclatFIM.FASHION_SOURCE,
    "julia_version" => string(VERSION),
    "image_shape" => [28, 28],
    "pixel_type" => string(eltype(train.images)),
    "checks_passed" => true,
    "train_images" => size(train.images, 3),
    "test_images" => size(test.images, 3),
    "train_class_counts" => train_counts,
    "test_class_counts" => test_counts,
    "archives" => [Dict("filename" => file.name, "sha256" => file.sha256)
        for file in expected_files],
)
report_path = joinpath(results_dir, "dataset_validation.toml")
open(report_path, "w") do io
    TOML.print(io, report; sorted=true)
end
println("Báo cáo: ", relpath(report_path, repo_root))


Báo cáo: results/fashion/dataset_validation.toml

Dataset đã được xác minh và có thể đọc qua `load_fashion_mnist(data_dir; split=:train)` hoặc `split=:test`. Notebook EDA tiếp theo tải split độc lập và hiển thị ảnh mẫu cùng thống kê theo lớp.